# 🚀 The Full SQD Pipeline

### *A Qiskit Fall Fest lab challenge celebrating 10 years of IBM Quantum in the cloud* 🎉

In May 2016, IBM put the first real quantum computer on the cloud. A decade later,
this is the culminating lab: you run the same workflow behind today's record-setting
quantum chemistry experiments, end to end.

**SQD from scratch, core notebook 3 of 3** · ⏱ about 75 minutes

Time to assemble everything. In this notebook you run the **production SQD workflow**,
the same shape as the experiments on real IBM Quantum hardware, on a molecule with a
genuinely difficult electronic structure: **N₂ with its triple bond stretched**.

```
   1                     2                       3                   4
 molecule  ──────►  ansatz circuit  ──────►  samples  ──────►  configuration recovery
 + integrals        (ffsim, from            (Aer here,         + crop + diagonalize
 (pyscf)            CCSD's guess)           QPU in real        (qiskit-addon-sqd)
                                            life)                      │
                                                                       ▼
                                                                     energy
```

This map previews the whole journey, so several of its labels are terms you have not
met yet: *ansatz*, *ffsim*, *CCSD*, *configuration recovery*. That is intentional.
Each numbered section defines its own vocabulary when you arrive at it; nothing
on the map will be left unexplained. (The labels you already own from notebooks 1
and 2: integrals, crop, diagonalize, and `pyscf`.)

**🎯 By the end you will be able to:**
- build an **ansatz** (a "starting guess" circuit) for a molecule straight from a
  standard classical chemistry calculation (CCSD), no new theory needed,
- sample it, apply a simple post-sampling bit-flip model, and compare one-pass
  with multi-pass *configuration recovery*,
- run the whole algorithm with one call: `diagonalize_fermionic_hamiltonian`,
- report sampled-row counts separately from determinant subspace dimensions,
- reuse a short `run_sqd()` coordinator on molecules of your choice.

**You should already know:** notebooks 1 and 2. The simulation and repeated
classical solves are the computationally heavy part of the series.

In [ ]:
# If you are on Google Colab or a fresh environment, uncomment and run:
# %pip install "numpy>=2.3.2" "qiskit==2.5.1" "qiskit-aer==0.17.2" \
#     "qiskit-addon-sqd==0.12.1" "ffsim>=0.0.81" "pyscf==2.14.0" \
#     "matplotlib==3.11.1" "pylatexenc==2.10"

from collections import Counter
from math import comb

import numpy as np
import matplotlib.pyplot as plt
import pyscf
import pyscf.ao2mo
import pyscf.cc
import pyscf.mcscf
import ffsim
from qiskit import QuantumCircuit
from qiskit.primitives import BitArray
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import SamplerV2
from qiskit_addon_sqd.fermion import diagonalize_fermionic_hamiltonian

rng = np.random.default_rng(2026)

Four chemistry methods appear repeatedly. Keep this fixed reference table nearby:

| Method | What it contributes here |
|---|---|
| Hartree-Fock (HF) | Optimized orbitals and the dominant determinant |
| CCSD | Classical one- and two-electron hop amplitudes used to initialize the circuit |
| FCI | Small-system answer key only |
| SQD | Frequency-aware recovery plus projected diagonalization in sampled determinant subspaces |

## 1 · Molecule + integrals

Nitrogen, N₂, holds the strongest bond in everyday chemistry: a triple bond. Real
molecules stretch as they vibrate and, more strongly, along reaction paths or after
absorbing enough energy to begin breaking a bond. We'll stretch N₂ from its natural
1.09 Å to **1.5 Å**, a controlled snapshot halfway to broken. Notebook 2 taught you what
that means: several configurations become important at once, and single-chart methods
get into serious trouble. This is precisely the regime SQD is built for.

As in notebook 2, the goal of this section is the **integrals** `hcore` and `eri`: the
compact kit of numbers from which any entry of the molecule's $H$ can be built. Two
economies keep N₂'s 14 electrons laptop-friendly. First, a smaller basis set:
**STO-3G**, the most compact in standard use (notebook 2 used the larger 6-31G).
Second, a new trick: we **freeze** the two innermost orbitals (core electrons never
leave the nucleus's side, so we lock their seats and stop simulating them). Freezing
2 orbitals removes their 4 electrons from the problem: 14 - 4 = 10 electrons in the
remaining 8 orbitals, 5 alpha and 5 beta, so **16 qubits**. Chemists call the
remaining orbitals the **active** orbitals.

<details>
<summary>🔎 <b>Under the hood: the active-space helper</b></summary>

One honest warning about the next cell: because of the frozen core, the code looks
*different* from notebook 2's, and it uses pyscf machinery we will not explain
(`CASCI`, short for "complete active space configuration interaction", and its
helpers). Treat it as a black-box incantation with a simple contract: molecule and
frozen-orbital count in; `hcore`, `eri`, and `e_core` for the active orbitals out.

</details>

Alongside the integrals we compute three classical reference energies. Two are old
friends: Hartree-Fock and FCI. The third is **CCSD** (*coupled cluster singles and
doubles*), chemistry's workhorse approximation: starting from Hartree-Fock, it
estimates the corrections from every one- and two-electron hop into empty seats. We
compute it here for its energy, and, more importantly, for section 2: the numbers CCSD
produces along the way will wire our quantum circuit.

In [ ]:
def build_active_space_problem(atom, basis="sto-3g", n_frozen=0, compare_to_fci=True):
    """Build a small closed-shell active-space chemistry problem."""
    mol = pyscf.gto.Mole()
    mol.build(atom=atom, basis=basis, verbose=0)
    assert mol.nelectron % 2 == 0, "this teaching helper expects a closed-shell molecule"

    scf = pyscf.scf.RHF(mol).run()
    active_orbitals = list(range(n_frozen, mol.nao_nr()))
    norb = len(active_orbitals)
    n_active_electrons = int(round(np.sum(scf.mo_occ[active_orbitals])))
    n_alpha = (n_active_electrons + mol.spin) // 2
    n_beta = (n_active_electrons - mol.spin) // 2
    nelec = (n_alpha, n_beta)

    # CASCI is used here only as a convenient active-space integral transformer.
    cas = pyscf.mcscf.CASCI(scf, norb, nelec)
    mo = cas.sort_mo(active_orbitals, base=0)
    hcore, e_core = cas.get_h1cas(mo)
    eri = pyscf.ao2mo.restore(1, cas.get_h2cas(mo), norb)

    frozen = [index for index in range(mol.nao_nr()) if index not in active_orbitals] or None
    ccsd = pyscf.cc.CCSD(scf, frozen=frozen).run()
    e_fci = cas.kernel(mo)[0] if compare_to_fci else None

    return {
        "atom": atom,
        "basis": basis,
        "n_frozen": n_frozen,
        "mol": mol,
        "scf": scf,
        "ccsd": ccsd,
        "norb": norb,
        "nelec": nelec,
        "hcore": hcore,
        "eri": eri,
        "e_core": e_core,
        "e_hf": scf.e_tot,
        "e_ccsd": ccsd.e_tot,
        "e_fci": e_fci,
    }


def print_problem_summary(problem):
    print(f"active orbitals: {problem['norb']} | electrons: {problem['nelec']} | "
          f"qubits: {2 * problem['norb']}")
    if problem["e_fci"] is not None:
        reference = problem["e_fci"]
        print(f"Hartree-Fock : {problem['e_hf']:.6f} Ha | error {(problem['e_hf'] - reference) * 1000:+8.2f} mHa")
        print(f"CCSD         : {problem['e_ccsd']:.6f} Ha | error {(problem['e_ccsd'] - reference) * 1000:+8.2f} mHa")
        print(f"exact (FCI)  : {reference:.6f} Ha")
    else:
        print(f"Hartree-Fock : {problem['e_hf']:.6f} Ha")
        print(f"CCSD         : {problem['e_ccsd']:.6f} Ha")


bond = 1.5
n2_atom = [["N", (0, 0, 0)], ["N", (bond, 0, 0)]]
problem = build_active_space_problem(n2_atom, basis="sto-3g", n_frozen=2, compare_to_fci=True)
print_problem_summary(problem)

norb = problem["norb"]
nelec = problem["nelec"]
n_alpha, n_beta = nelec
hcore, eri, e_core = problem["hcore"], problem["eri"], problem["e_core"]
e_hf, e_ccsd, e_fci = problem["e_hf"], problem["e_ccsd"], problem["e_fci"]

**🧠 Checkpoint 1.** 16 qubits means $2^{16} = 65{,}536$ bitstrings. How many of them are
*valid* configurations (5 alpha seats among 8 orbitals, 5 beta among 8)? What fraction
is that?

<details>
<summary>💡 <b>Check your answer</b></summary>

Count each spin's seating choices separately. The 5 alpha electrons need 5 of the 8
orbitals; the number of ways to choose them is written $\binom{8}{5}$, read "8 choose
5", and equals 56. (No formula needed: `math.comb(8, 5)` in Python.) The beta
electrons independently make their own choice from the same 56 options, so the counts
multiply:

$$\binom{8}{5}^2 = 56 \times 56 = 3{,}136$$

valid configurations, under **5%** of all 65,536 bitstrings. Two consequences: (1) a
good ansatz should spend its probability only on that 5%, and (2) noise, which flips
bits indiscriminately, will constantly knock samples *out* of that 5%. Keep both in
mind; they drive the rest of this notebook.

*(A note on scale: 3,136 configurations is classically trivial, which is exactly why we can
grade ourselves against FCI here. The identical pipeline has run with tens of millions of
configurations on 77 qubits of real hardware, far beyond exact diagonalization.)*

</details>

## 2 · The ansatz: turn CCSD's guess into a circuit

*Where we are: section 1 produced the molecule's rulebook (`hcore`, `eri`, `e_core`) and
three reference energies. What we lack is something to sample.*

We need a circuit whose samples point at the right configurations. Quantum folks call
such a circuit an **ansatz** (German for "starting guess"): a circuit whose whole job
is to prepare an educated first draft of the state you're after. Perfection not
required, as notebook 1 proved.

The educated part comes from CCSD, which you met among section 1's reference energies.
Beyond its energy, CCSD records how strongly each of those one- and two-electron hops
wants to happen; the hop strengths are the amplitudes `t1` and `t2` it computed for
us. Think of them as a classically computed shortlist of the most promising electron
moves.

Turning that shortlist into a circuit is a solved problem, and `ffsim` is the
factory. The recipe is the **UCJ ansatz** (short for *unitary cluster
Jastrow*), in three bullets:

- start from the Hartree-Fock bitstring (that's `PrepareHartreeFockJW`: literally X gates
  on the occupied seats),
- apply an "electron shuffler" whose rotation angles are read directly off the CCSD
  amplitudes we already computed. CCSD's classical wisdom about *which pair-hops matter*
  becomes the circuit's wiring: no training loop, no optimizer,
- the shuffler **cannot change the electron count** of either spin. Every noiseless
  sample is automatically one of Checkpoint 1's valid 5%.

`n_reps=2` means two passes of the shuffler: a slightly richer guess.

<details>
<summary>🔎 <b>Optional naming detail: UCJ versus LUCJ</b></summary>

(One naming
note: the class below says `UCJ`; the *local* restriction that puts the L in LUCJ is
a hardware-focused adaptation. On this simulator we use the unrestricted version.)

</details>

In [ ]:
def build_ansatz(problem, n_reps=2, measure=True):
    """Build a spin-balanced UCJ ansatz initialized from the problem's CCSD amplitudes."""
    norb = problem["norb"]
    nelec = problem["nelec"]
    ccsd = problem["ccsd"]

    ucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
        t2=ccsd.t2,
        t1=ccsd.t1,
        n_reps=n_reps,
    )
    circuit = QuantumCircuit(2 * norb)
    circuit.append(ffsim.qiskit.PrepareHartreeFockJW(norb, nelec), circuit.qubits)
    circuit.append(ffsim.qiskit.UCJOpSpinBalancedJW(ucj_op), circuit.qubits)
    if measure:
        circuit.measure_all()
    return circuit


circuit = build_ansatz(problem, n_reps=2, measure=True)
circuit.draw("mpl", fold=-1)

**🧠 Checkpoint 2.** Qubits 0-7 carry the alpha seats and qubits 8-15 the beta seats, so a
measured bitstring reads `beta_bits + alpha_bits`, exactly notebook 2's `config()` format.
(The `JW` tag on both circuit pieces names this seat-to-qubit dictionary: the
**Jordan-Wigner** mapping, the standard convention.) Why is it so valuable that the
ansatz conserves each spin's electron count?

<details>
<summary>💡 <b>Check your answer</b></summary>

Because only ~5% of bitstrings are valid configurations. A generic circuit (say, a
random hardware-efficient ansatz) would scatter probability over the invalid 95%,
wasting most shots. Number conservation channels **every** shot into the physically
meaningful subspace. Whatever noise later breaks will at least have started correct.

</details>

## 3 · Sample it

*Where we are: a molecule with a rulebook (section 1) and a circuit that points at its
important configurations (section 2). Time to collect bitstrings.*

The helper below compiles the circuit for Aer, samples it, and returns both the
`BitArray` needed by SQD and readable strings for inspection.

<details>
<summary>🔎 <b>Optional compiler vocabulary</b></summary>

Our circuit is written with ffsim's high-level building blocks, so before running we
let Qiskit's compiler rewrite it in the backend's own gate language. Current best
practice is a two-liner: `generate_preset_pass_manager` builds a reusable compiler
pipeline tuned to a backend, and its `run` method does the rewriting. (The result is
called an **ISA circuit**: one that uses only instructions the backend natively
supports.) On real hardware you'll type the exact same two lines, just with a real
backend; today the QPU is played by Aer. (~20 seconds: it's a real 16-qubit simulation.)

</details>

In [ ]:
def sample_circuit(circuit, shots=20_000, seed=123, optimization_level=1):
    """Compile and sample a measured circuit on Aer."""
    backend = AerSimulator()
    pass_manager = generate_preset_pass_manager(
        optimization_level=optimization_level,
        backend=backend,
    )
    isa_circuit = pass_manager.run(circuit)
    two_qubit_gates = sum(
        1 for instruction in isa_circuit.data if instruction.operation.num_qubits == 2
    )
    bit_array = (
        SamplerV2(seed=seed)
        .run([isa_circuit], shots=shots)
        .result()[0]
        .data.meas
    )
    strings = bit_array.get_bitstrings()
    return {
        "backend": backend,
        "isa_circuit": isa_circuit,
        "bit_array": bit_array,
        "strings": strings,
        "shots": shots,
        "depth": isa_circuit.depth(),
        "two_qubit_gates": two_qubit_gates,
    }


sample_record = sample_circuit(circuit, shots=20_000, seed=123)
bit_array = sample_record["bit_array"]
strings = sample_record["strings"]
shots = sample_record["shots"]

print(f"compiled circuit: depth {sample_record['depth']}, "
      f"{sample_record['two_qubit_gates']} two-qubit gates")
print(f"{bit_array.num_shots:,} shots x {bit_array.num_bits} bits, "
      f"{len(set(strings)):,} distinct sampled rows")

In [ ]:
def bitstrings_to_bool(strings):
    return np.asarray([[character == "1" for character in string] for string in strings], dtype=bool)


def valid_electron_counts(bitstrings, norb, nelec):
    """Mask rows with the target alpha count on the right and beta count on the left."""
    bitstrings = np.asarray(bitstrings, dtype=bool)
    alpha_ok = bitstrings[:, norb:].sum(axis=1) == nelec[0]
    beta_ok = bitstrings[:, :norb].sum(axis=1) == nelec[1]
    return alpha_ok & beta_ok


def pretty(s):
    return s[:norb] + "·" + s[norb:]


bools_clean = bitstrings_to_bool(strings)
clean_valid = valid_electron_counts(bools_clean, norb, nelec)
assert clean_valid.all(), "the noiseless number-conserving ansatz should return only valid rows"
print(f"valid electron counts: {clean_valid.mean():.0%}")

In [ ]:
def pretty(s):
    return s[:norb] + "·" + s[norb:]  # visual split: beta·alpha


hf_string = "0" * (norb - n_alpha) + "1" * n_alpha
hf_full = hf_string + hf_string

top = Counter(strings).most_common(20)
colors = ["#fa4d56" if s == hf_full else "#4589ff" for s, _ in top]
fig, ax = plt.subplots(figsize=(10, 3.8))
ax.bar(range(len(top)), [c for _, c in top], color=colors)
ax.set_yscale("log")
ax.set_xticks(range(len(top)))
ax.set_xticklabels([pretty(s) for s, _ in top], rotation=90, fontsize=7, family="monospace")
ax.set_ylabel(f"shots (of {shots:,}, log)")
ax.set_title("the quantum computer's shortlist (red = the Hartree-Fock configuration)")
plt.tight_layout()
plt.show()

There it is: the concentration silhouette from notebooks 1 and 2, now produced by an
actual circuit. Every bar is a valid 5-alpha/5-beta configuration, and the heavy ones
are precisely the configurations stretched N₂ needs.

> **Support and frequency now have different jobs.** The set of distinct rows bounds
> which configurations were directly observed. The number of times each row appeared
> estimates a probability distribution, which configuration recovery uses to infer
> average orbital occupancies. Do not deduplicate the `BitArray` before section 4.

## Reality check · a simple post-sampling bit-flip model

The next experiment is deliberately limited: after sampling, flip every recorded bit
independently with probability 2%. This is a readout-like toy model for studying broken
electron counts. It is **not** a complete model of a QPU, which can also have coherent,
correlated, relaxation, and crosstalk errors.

In [ ]:
def add_bit_flip_noise(bit_array, probability, seed=2026):
    """Apply independent post-sampling bit flips and return a new BitArray plus booleans."""
    clean = bitstrings_to_bool(bit_array.get_bitstrings())
    local_rng = np.random.default_rng(seed)
    noisy = clean ^ (local_rng.random(clean.shape) < probability)
    return BitArray.from_bool_array(noisy, order="big"), noisy


def fixed_count_probability(norb, n_occupied, probability):
    """Probability that one spin sector keeps its Hamming weight after independent flips."""
    n_empty = norb - n_occupied
    return sum(
        comb(n_occupied, k)
        * comb(n_empty, k)
        * probability ** (2 * k)
        * (1 - probability) ** (norb - 2 * k)
        for k in range(min(n_occupied, n_empty) + 1)
    )


def valid_count_probability(norb, nelec, probability):
    return (
        fixed_count_probability(norb, nelec[0], probability)
        * fixed_count_probability(norb, nelec[1], probability)
    )


noise_p = 0.02
noisy_bit_array, noisy_bools = add_bit_flip_noise(bit_array, noise_p, seed=2026)
noisy_valid = valid_electron_counts(noisy_bools, norb, nelec)
electrons_measured = noisy_bools.sum(axis=1)

p_untouched = (1 - noise_p) ** (2 * norb)
p_valid_theory = valid_count_probability(norb, nelec, noise_p)
print(f"probability no bit flips at all:       {p_untouched:.1%}")
print(f"probability electron counts stay valid: {p_valid_theory:.1%} (exact model calculation)")
print(f"observed valid fraction:                {noisy_valid.mean():.1%}")

fig, ax = plt.subplots(figsize=(8, 3.6))
counts_w = np.bincount(electrons_measured, minlength=2 * norb + 1)
bars = ax.bar(range(len(counts_w)), counts_w, color="#8a3ffc")
bars[n_alpha + n_beta].set_color("#fa4d56")
ax.set_xlim(4.5, 15.5)
ax.set_xlabel(f"total electrons counted (target = {n_alpha + n_beta})")
ax.set_ylabel("samples")
ax.set_title(f"after {noise_p:.0%} independent post-sampling flips, "
             f"{noisy_valid.mean():.0%} retain both spin counts")
plt.tight_layout()
plt.show()

**🧠 Checkpoint 3.** Why are the probability of an untouched string and the
probability of a valid electron count different? Why does discarding every invalid
row still age badly as systems grow?

<details>
<summary>💡 <b>Check your answer</b></summary>

$(1-p)^n$ is the probability that **no bit flips at all**. A row can suffer two
compensating flips in the same spin sector, one occupied seat turning off and one empty
seat turning on, and still retain the correct electron count. The exact validity
probability therefore sums over all equal numbers of occupied-to-empty and
empty-to-occupied flips, as the helper above does.

Even so, more qubits create more chances to leave the target sector. Discarding invalid
rows wastes partially correct information and reduces the effective sample budget.
Configuration recovery instead uses the observed frequency distribution and orbital
occupancy estimates to repair candidate rows.

</details>

## 4 · Configuration recovery and projected diagonalization

*Where we are: the same circuit produced a clean `BitArray` and a toy-noisy one. We
now vary only the recovery depth while holding all other solver settings fixed.*

Here is the idea that rescues the noisy samples, and it is SQD's distinctive
strength. The valid samples, taken together, tell us the average occupancy of every
orbital: how often each seat is filled. **Configuration recovery** uses that
knowledge to *repair* invalid samples instead of discarding them: flip the bits that
are least plausible given the expected occupancies, until the electron counts come
out right. Then iterate: solve in the repaired subspace, read off better occupancies,
repair again. Broken samples become fresh, valid, useful configurations.

A repaired sample is not restricted to configurations anyone actually measured:
every repair can produce a valid configuration that was absent from the observed
support. Recovery can therefore enlarge the candidate pool as it repairs broken rows.
That exploratory effect does **not** make noise beneficial. The controlled comparison
below separates the cost of the toy noise from the effect of additional recovery
passes.

`diagonalize_fermionic_hamiltonian` is the complete iterative SQD entry point. Every
iteration uses sample frequencies to estimate occupancies, repairs candidate
configurations, splits them into batches, and diagonalizes the Hamiltonian in each
batch's alpha/beta determinant product space.

The helper below records, at every iteration:

- the best total energy across batches,
- each batch's actual `SCIState.amplitudes.shape`,
- the corresponding determinant dimension $M \times N$.

A one-iteration call is called **one recovery-and-solve pass**, not “no recovery.”
The current API performs recovery as part of that first iteration.

In [ ]:
def recover_and_diagonalize(
    problem,
    bit_array,
    *,
    label,
    samples_per_batch=500,
    num_batches=3,
    max_iterations=6,
    symmetrize_spin=False,
    seed=31415,
    verbose=True,
):
    """Run SQD and return a transparent record of energies and subspace dimensions."""
    history = []

    def callback(results):
        best = min(results, key=lambda result: result.energy)
        batch_shapes = [tuple(int(x) for x in result.sci_state.amplitudes.shape) for result in results]
        batch_dimensions = [int(np.prod(shape)) for shape in batch_shapes]
        total_energy = best.energy + problem["e_core"]
        history.append({
            "energy": total_energy,
            "batch_shapes": batch_shapes,
            "batch_dimensions": batch_dimensions,
        })
        if verbose:
            reference_text = ""
            if problem["e_fci"] is not None:
                reference_text = f" | error {(total_energy - problem['e_fci']) * 1000:+7.2f} mHa"
            print(f"  {label}, iteration {len(history)}: E = {total_energy:.6f} Ha{reference_text} "
                  f"| batch dimensions {batch_dimensions}")

    result = diagonalize_fermionic_hamiltonian(
        problem["hcore"],
        problem["eri"],
        bit_array,
        samples_per_batch=samples_per_batch,
        norb=problem["norb"],
        nelec=problem["nelec"],
        num_batches=num_batches,
        energy_tol=0.0,
        occupancies_tol=0.0,
        max_iterations=max_iterations,
        symmetrize_spin=symmetrize_spin,
        callback=callback,
        seed=seed,
    )

    energy = result.energy + problem["e_core"]
    final_shape = tuple(int(x) for x in result.sci_state.amplitudes.shape)
    strings = bit_array.get_bitstrings()
    error_mha = None if problem["e_fci"] is None else (energy - problem["e_fci"]) * 1000
    return {
        "label": label,
        "result": result,
        "energy": energy,
        "error_mha": error_mha,
        "history": history,
        "distinct_sample_rows": len(set(strings)),
        "final_shape": final_shape,
        "final_dimension": int(np.prod(final_shape)),
        "settings": {
            "samples_per_batch": samples_per_batch,
            "num_batches": num_batches,
            "max_iterations": max_iterations,
            "symmetrize_spin": symmetrize_spin,
            "seed": seed,
        },
    }

### Controlled four-condition experiment

The settings below are identical in all four conditions except for the two factors
named in the table:

| Condition | Samples | Recovery-and-solve passes |
|---|---|---:|
| A | clean | 1 |
| B | clean | 6 |
| C | toy-noisy | 1 |
| D | toy-noisy | 6 |

`num_batches`, `samples_per_batch`, `symmetrize_spin`, and the random seed are fixed.
This lets us isolate causal comparisons rather than changing several algorithmic knobs
at once.

In [ ]:
CONTROL_SETTINGS = {
    "samples_per_batch": 500,
    "num_batches": 3,
    "symmetrize_spin": False,
    "seed": 31415,
}

conditions = {}
conditions["A"] = recover_and_diagonalize(
    problem, bit_array, label="A clean, one pass", max_iterations=1, **CONTROL_SETTINGS
)
conditions["B"] = recover_and_diagonalize(
    problem, bit_array, label="B clean, six passes", max_iterations=6, **CONTROL_SETTINGS
)
conditions["C"] = recover_and_diagonalize(
    problem, noisy_bit_array, label="C noisy, one pass", max_iterations=1, **CONTROL_SETTINGS
)
conditions["D"] = recover_and_diagonalize(
    problem, noisy_bit_array, label="D noisy, six passes", max_iterations=6, **CONTROL_SETTINGS
)

In [ ]:
def print_condition_table(conditions):
    print("\ncondition | distinct sampled rows | final alpha x beta shape | determinant dimension | error (mHa)")
    print("-" * 103)
    for key in ["A", "B", "C", "D"]:
        item = conditions[key]
        error_text = "n/a" if item["error_mha"] is None else f"{item['error_mha']:+10.3f}"
        print(f"{key:^9} | {item['distinct_sample_rows']:^21,} | "
              f"{str(item['final_shape']):^24} | {item['final_dimension']:^21,} | {error_text}")


print_condition_table(conditions)

fig, ax = plt.subplots(figsize=(8.2, 4.2))
for key in ["B", "D"]:
    history_errors = [max((entry["energy"] - e_fci) * 1000, 1e-10)
                      for entry in conditions[key]["history"]]
    ax.semilogy(range(1, len(history_errors) + 1), history_errors, "o-",
                label=conditions[key]["label"])
ax.axhline(1.6, color="green", ls="--", label="chemical accuracy")
ax.set_xlabel("recovery-and-solve iteration")
ax.set_ylabel("energy error vs FCI (mHa, log)")
ax.set_title("extra recovery passes, with clean and toy-noisy samples")
ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8.2, 3.8))
keys = ["A", "B", "C", "D"]
errors = [max(abs(conditions[key]["error_mha"]), 1e-10) for key in keys]
bars = ax.bar(keys, errors)
ax.set_yscale("log")
ax.axhline(1.6, color="green", ls="--", label="chemical accuracy")
for bar, key in zip(bars, keys):
    value = conditions[key]["error_mha"]
    ax.text(bar.get_x() + bar.get_width() / 2, max(abs(value), 1e-10) * 1.25,
            f"{value:+.2f}", ha="center", fontsize=9)
ax.set_xlabel("controlled condition")
ax.set_ylabel("|error| vs FCI (mHa, log)")
ax.set_title("same batching and symmetry settings in all four conditions")
ax.legend()
plt.tight_layout()
plt.show()

clean_recovery_gain = conditions["A"]["error_mha"] - conditions["B"]["error_mha"]
noisy_recovery_gain = conditions["C"]["error_mha"] - conditions["D"]["error_mha"]
print(f"extra-pass improvement on clean samples (A - B): {clean_recovery_gain:+.3f} mHa")
print(f"extra-pass improvement on noisy samples (C - D): {noisy_recovery_gain:+.3f} mHa")
if conditions["D"]["error_mha"] < conditions["B"]["error_mha"]:
    print("In this seeded run, repaired noisy data finishes below repaired clean data.")
    print("Treat that as an observed outcome, not evidence that noise is generally beneficial.")
else:
    print("In this seeded run, repaired clean data finishes at least as accurately as repaired noisy data.")

Read the comparisons one factor at a time:

- **A versus B** isolates the effect of additional recovery passes on clean samples.
- **C versus D** isolates the same effect under the toy-noisy samples.
- **A versus C** isolates the toy noise after one pass.
- **B versus D** compares the two final workflows, but should not be interpreted as
  “noise helps” or “noise hurts” without the first three comparisons.

Recovery can mint valid configurations that were never directly measured, so a noisy
run can occasionally finish below a clean run for a particular seed and budget. That
is an empirical outcome of the exploration process, not a general causal benefit of
noise.

The table also separates two counts that are easy to confuse: **distinct sampled full
rows** and the final solver's **alpha x beta determinant dimension**.

**🧠 Checkpoint 4.** Which pair tells you whether recovery helped noisy data, and which
property guarantees every SQD energy remains an upper bound to FCI?

<details>
<summary>💡 <b>Check your answer</b></summary>

Compare **C with D**: the samples, batching, symmetry choice, and seed are fixed; only
the number of recovery-and-solve passes changes. SQD remains **variational** because
each energy is obtained by diagonalizing the true Hamiltonian in a restricted
subspace. Better subspaces can lower the answer toward FCI, but not below it except for
small numerical roundoff.

</details>

### Optional optimization · spin symmetrization

The controlled experiment used `symmetrize_spin=False` so alpha and beta sectors kept
the transparent semantics established in notebook 2. For a closed-shell molecule,
`True` merges the recovered alpha and beta CI-string sets before diagonalization,
expanding the subspace using known spin-exchange symmetry.

This is a separate algorithmic choice, so the optional cell is off by default rather
than silently changing the A-D comparison.

In [ ]:
RUN_SPIN_SYMMETRY_COMPARISON = False

if RUN_SPIN_SYMMETRY_COMPARISON:
    symmetric_D = recover_and_diagonalize(
        problem,
        noisy_bit_array,
        label="noisy, six passes, spin-symmetrized",
        samples_per_batch=CONTROL_SETTINGS["samples_per_batch"],
        num_batches=CONTROL_SETTINGS["num_batches"],
        max_iterations=6,
        symmetrize_spin=True,
        seed=CONTROL_SETTINGS["seed"],
    )
    print(f"without spin symmetrization: {conditions['D']['error_mha']:+.3f} mHa, "
          f"dimension {conditions['D']['final_dimension']:,}")
    print(f"with spin symmetrization:    {symmetric_D['error_mha']:+.3f} mHa, "
          f"dimension {symmetric_D['final_dimension']:,}")
else:
    print("Spin-symmetry comparison skipped. Set RUN_SPIN_SYMMETRY_COMPARISON = True to run it.")

### ✍️ Your turn 1 · raise the toy noise

Predict first: at a **5%** post-sampling flip rate, how will the valid-row fraction and
six-pass SQD error change? Use the two helpers rather than copying the low-level addon
call. Define `noisy5_bit_array` and `condition5`; the scoreboard handles the rest.

In [ ]:
# ✏️ =============== YOUR CODE HERE ===============
noisy5_bit_array = ...  # TODO: add_bit_flip_noise(bit_array, 0.05, seed=2027)[0]
condition5 = ...        # TODO: recover_and_diagonalize(..., max_iterations=6, **CONTROL_SETTINGS)
# ✏️ ============== END OF YOUR CODE ==============
assert noisy5_bit_array is not Ellipsis and condition5 is not Ellipsis, \
    "✏️ Build the 5% BitArray and run it through recover_and_diagonalize first."

noisy5_bools = bitstrings_to_bool(noisy5_bit_array.get_bitstrings())
valid5 = valid_electron_counts(noisy5_bools, norb, nelec).mean()
print(f"at 5% noise: {valid5:.0%} of rows retain both electron counts")
print(f"six-pass SQD error: {condition5['error_mha']:+.3f} mHa")
print("within chemical accuracy" if abs(condition5["error_mha"]) < 1.6
      else "above chemical accuracy in this seeded run")

<details>
<summary>💡 <b>Solution: Your turn 1</b></summary>

```python
noisy5_bit_array, _ = add_bit_flip_noise(bit_array, 0.05, seed=2027)
condition5 = recover_and_diagonalize(
    problem,
    noisy5_bit_array,
    label="5% toy noise, six passes",
    max_iterations=6,
    **CONTROL_SETTINGS,
)
```

The exact outcome depends on the fixed samples and seed. The controlled question is
whether six passes improve over a one-pass 5% condition, not whether every noise level
must reach a predetermined accuracy threshold.

</details>

## The repeatable recipe · `run_sqd()`

The four stage helpers already mirror the pipeline diagram:

1. `build_active_space_problem`
2. `build_ansatz`
3. `sample_circuit`
4. `recover_and_diagonalize`

The coordinator below is intentionally short. The N₂ work above has already exercised
each stage, so we do **not** repeat that expensive molecule as a “sanity check.”

In [ ]:
def run_sqd(
    atom,
    *,
    basis="sto-3g",
    n_frozen=0,
    n_reps=2,
    shots=20_000,
    noise_p=0.02,
    samples_per_batch=500,
    num_batches=3,
    max_iterations=6,
    symmetrize_spin=False,
    seed=2026,
    compare_to_fci=True,
):
    """Coordinate the four teaching-stage helpers for a small closed-shell molecule."""
    problem = build_active_space_problem(
        atom, basis=basis, n_frozen=n_frozen, compare_to_fci=compare_to_fci
    )
    print_problem_summary(problem)
    circuit = build_ansatz(problem, n_reps=n_reps, measure=True)
    samples = sample_circuit(circuit, shots=shots, seed=seed)

    solve_samples = samples["bit_array"]
    noisy_bools = None
    if noise_p:
        solve_samples, noisy_bools = add_bit_flip_noise(solve_samples, noise_p, seed=seed + 1)

    sqd = recover_and_diagonalize(
        problem,
        solve_samples,
        label="run_sqd",
        samples_per_batch=samples_per_batch,
        num_batches=num_batches,
        max_iterations=max_iterations,
        symmetrize_spin=symmetrize_spin,
        seed=seed + 2,
    )
    return {
        "problem": problem,
        "circuit": circuit,
        "samples": samples,
        "noisy_bools": noisy_bools,
        "sqd": sqd,
    }

### ✍️ Your turn 2 · your first *own* SQD example

The capstone. Run the recipe on **water**, geometry below, freezing 1 core orbital
(oxygen's innermost). Before running, predict: how many qubits will it use?
*(Water in STO-3G has 7 orbitals; you're freezing 1.)*

```python
water = [["O", (0.0, 0.0, 0.1173)],
         ["H", (0.0, 0.7572, -0.4692)],
         ["H", (0.0, -0.7572, -0.4692)]]
```

In [ ]:
water = [["O", (0.0, 0.0, 0.1173)],
         ["H", (0.0, 0.7572, -0.4692)],
         ["H", (0.0, -0.7572, -0.4692)]]

# ✏️ =============== YOUR CODE HERE ===============
# TODO: call run_sqd on water with 1 frozen orbital: h2o = run_sqd(...)
# ✏️ ============== END OF YOUR CODE ==============

<details>
<summary>💡 <b>Solution: Your turn 2</b></summary>

```python
h2o = run_sqd(atom=water, n_frozen=1)
```

6 active orbitals → **12 qubits**. Inspect `h2o["sqd"]["final_shape"]` and
`h2o["sqd"]["final_dimension"]` rather than inferring the determinant dimension
from the number of distinct sampled full rows.

You just created an SQD example from scratch: that's the repeatable process. Ideas to
explore with the same one-liner: stretch N₂ further (`2.0` Å: harder correlation),
stretch water's O-H bonds, raise `noise_p` until recovery breaks, or lower `shots` until
sampling starves. Try `compare_to_fci=False` to see what the printout looks like at
real scale, where no answer key exists. Stay modest on size: every extra orbital adds
2 simulated qubits, and a laptop simulator taps out around 24.

</details>

🎉 **The core lab ends here.** You have run the complete SQD pipeline and built your
own example with it; the series' goals are met.


## Beyond this core sequence

For a maintained example of real-hardware execution, the
[official SQD chemistry tutorial](https://qiskit.github.io/qiskit-addon-sqd/tutorials/01_chemistry_hamiltonian.html)
shows the current hardware-oriented workflow.

<details>
<summary>🔎 <b>Optional closing vocabulary: the Qiskit pattern</b></summary>

One last piece of vocabulary, now that you have lived it: IBM's documentation calls
the four-step shape you just ran a **Qiskit pattern**: *map* the problem to circuits
and operators (sections 1-2), *optimize* for the target hardware (section 3's compile),
*execute* with a **primitive** such as the `Sampler` (section 3's shots), and
*post-process* (section 4). Every serious Qiskit workflow follows this shape; you now
recognize it from the inside.

</details>

## 🎓 Series recap: the SQD recipe, version 1.0

1. **Molecule → active-space integrals** (`pyscf`): `hcore`, `eri`, `e_core`.
2. **CCSD → number-conserving ansatz** (`ffsim`): a chemistry-informed pointer at useful configurations.
3. **Sample** (`SamplerV2`): preserve both distinct support and frequencies.
4. **Configuration recovery + projected diagonalization**: iterate with fixed, explicit solver settings.
5. **Report two different sizes**: distinct sampled full rows and the alpha x beta determinant dimension.
6. **Add `e_core`** and compare against HF/CCSD, plus FCI only while the system is small enough.

And the three intuitions you can now defend at a whiteboard:

| Intuition | Where you proved it |
|---|---|
| Ground states concentrate on few bitstrings | magnet (NB1), H₂ (NB2), N₂ (NB3) |
| Subspace energies are safe upper bounds that only improve | NB1 crop, NB2 staircase, Checkpoint 4 |
| The QPU proposes, the classical computer disposes | NB1 central result, NB3 pipeline |

A final experimental habit matters just as much: when comparing workflows, vary one
factor at a time. The A-D design separated noise, recovery depth, batching, and spin
symmetry so the plotted differences had interpretable causes.

**Where to go next:**
- [qiskit-addon-sqd documentation](https://qiskit.github.io/qiskit-addon-sqd/)
- [ffsim documentation](https://qiskit-community.github.io/ffsim/)
- [Robledo-Moreno et al., *Science Advances*](https://arxiv.org/abs/2405.05068)

Happy diagonalizing! 🎉